# CA + MX Disaster Risk Data — Research & Gap Analysis

This notebook documents the data-source and integration challenges for generating
per-province/state risk CSVs for **Canada** and **Mexico** analogous to the US NRI CSVs
produced by `US_disaster_risk_analysis.ipynb`.

**This notebook does not generate any output files.** It demonstrates — with live API
calls — what existing services provide, where the gaps are, and what would be required
to close them.

## The core problem

The client-side JS in `emfn-behavior-plugin.js` works like this for US users:

```
Google Places v2 (address) → lat/lng
       ↓
FCC Area API (lat/lng) → county FIPS code     ← US-only
       ↓
{ST}.csv row lookup by FIPS code
```

For CA/MX there is no FCC equivalent. The most credible public hazard data for
both countries is **[ThinkHazard](https://thinkhazard.org/)** (GFDRR/World Bank), which
uses its own internal numeric division IDs derived from GADM. The question this notebook
investigates is: *can we get from a Google Places v2 response to a ThinkHazard division
ID without an additional geocoding service?*

## Sections

1. ThinkHazard API — what it provides and how division IDs work
2. Live CA division report
3. Live MX division report
4. Hazard taxonomy mapping (ThinkHazard mnemonics vs NRI codes)
5. What Google Places v2 returns for CA/MX addresses
6. FCC API — live demonstration that it is US-only
7. The lookup gap — why lat/lng or text name is not enough
8. What would close the gap

In [ ]:
import json
import pprint
from io import StringIO

import pandas as pd
import requests

TH_BASE = "https://thinkhazard.org/en"
FCC_BASE = "https://geo.fcc.gov/api/census/block/find"

## 1. ThinkHazard API

ThinkHazard provides classified hazard levels (VLO / LOW / MED / HIG) for admin-3
units worldwide. Its API has two relevant endpoints:

| Endpoint | Description |
|----------|-------------|
| `GET /en/report/{code}.json` | All hazard levels for one division |
| `GET /en/admindiv_hazardsets/{haz}.json` | All admin-3 divisions globally for one hazard type |

Division codes are **ThinkHazard-internal numeric IDs** derived from the
[GADM](https://gadm.org/) global administrative boundary dataset. They are not
ISO codes, postal abbreviations, or FIPS codes.

The cell below fetches the full admin-division list for earthquake (`EQ`) — one of
the smaller single-hazard files — and filters for Canada and Mexico to extract
representative division codes.

In [ ]:
# Fetch the global admin-3 division list for one hazard type.
# ~1–2 MB; cached in memory only — not written to disk.
print("Fetching ThinkHazard admin division list for EQ …")
resp = requests.get(f"{TH_BASE}/admindiv_hazardsets/EQ.json", timeout=60)
resp.raise_for_status()
all_divisions = resp.json()
print(f"Total admin-3 divisions globally: {len(all_divisions):,}")
print(f"\nSample record:")
pprint.pprint(all_divisions[0])

In [ ]:
# Filter to Canada and Mexico
ca_divs = [d for d in all_divisions if d.get("level_1") == "Canada"]
mx_divs = [d for d in all_divisions if d.get("level_1") == "Mexico"]

print(f"Canada admin-3 divisions (EQ data): {len(ca_divs)}")
print("\nFirst 5 CA divisions:")
for d in ca_divs[:5]:
    print(f"  code={d['code']:>6}  level_2={d['level_2']:<30}  name={d['name']:<30}  level={d['hazard_level']}")

print(f"\nMexico admin-3 divisions (EQ data): {len(mx_divs)}")
print("\nFirst 5 MX divisions:")
for d in mx_divs[:5]:
    print(f"  code={d['code']:>6}  level_2={d['level_2']:<30}  name={d['name']:<30}  level={d['hazard_level']}")

### Key observation from the division list

Each row has `name` (census division or municipio name), `level_2` (province/state
name), `level_1` (country), and **`code`** — a bare integer with no relation to any
public geographic identifier. There is no reverse-geocode endpoint on ThinkHazard;
the only way to resolve a location to a `code` is through this name hierarchy.

## 2. Live CA division report

Using a `code` retrieved above, fetch the full hazard report for one Canadian
census division. This shows what ThinkHazard can provide once you have the code.

In [ ]:
# Pick the first CA division with a non-VLO hazard level for a more interesting report.
ca_example = next(
    (d for d in ca_divs if d.get("hazard_level") not in ("VLO", None)),
    ca_divs[0],
)
print(f"Using CA division: {ca_example['name']} ({ca_example['level_2']})  code={ca_example['code']}")

ca_report = requests.get(f"{TH_BASE}/report/{ca_example['code']}.json", timeout=30)
ca_report.raise_for_status()
ca_hazards = ca_report.json()

print(f"\nHazard levels for '{ca_example['name']}':")
for entry in ca_hazards:
    htype = entry["hazardtype"]["mnemonic"]
    hlevel = entry["hazardlevel"]["mnemonic"]
    hname  = entry["hazardtype"]["hazardtype"]
    print(f"  {htype:>3}  {hlevel:<4}  {hname}")

## 3. Live MX division report

In [ ]:
mx_example = next(
    (d for d in mx_divs if d.get("hazard_level") not in ("VLO", None)),
    mx_divs[0],
)
print(f"Using MX division: {mx_example['name']} ({mx_example['level_2']})  code={mx_example['code']}")

mx_report = requests.get(f"{TH_BASE}/report/{mx_example['code']}.json", timeout=30)
mx_report.raise_for_status()
mx_hazards = mx_report.json()

print(f"\nHazard levels for '{mx_example['name']}':")
for entry in mx_hazards:
    htype = entry["hazardtype"]["mnemonic"]
    hlevel = entry["hazardlevel"]["mnemonic"]
    hname  = entry["hazardtype"]["hazardtype"]
    print(f"  {htype:>3}  {hlevel:<4}  {hname}")

## 4. Hazard taxonomy mapping

ThinkHazard covers **11 hazard types** using its own mnemonics. The NRI covers **18**.
The tables below show the overlap and the gaps.

| ThinkHazard mnemonic | ThinkHazard name | Nearest NRI code | NRI name |
|---------------------|-----------------|-----------------|----------|
| EQ | Earthquake | ERQK | Earthquake |
| FL | River flood | IFLD | Inland Flooding |
| UF | Urban/flash flood | IFLD | Inland Flooding |
| CF | Coastal flood | CFLD | Coastal Flooding |
| TS | Tsunami | TSUN | Tsunami |
| LS | Landslide | LNDS | Landslide |
| CY | Cyclone/Tropical storm | HRCN | Hurricane |
| EH | Extreme heat | HWAV | Heat Wave |
| DG | Drought | DRGT | Drought |
| VO | Volcanic activity | VLCN | Volcanic Activity |
| WF | Wildfire | WFIR | Wildfire |

**NRI codes with no ThinkHazard equivalent:** AVLN (Avalanche), CWAV (Cold Wave),
HAIL, ISTM (Ice Storm), LTNG (Lightning), SWND (Strong Wind), TRND (Tornado), WNTW (Winter Weather).

Those 8 NRI-only types are US-centric meteorological hazards. For CA/MX output CSVs
these columns would either be omitted or left null.

In [ ]:
# ThinkHazard level mnemonics → numeric proxy scores (for CSV compatibility)
# These are approximate midpoints of the implied percentile ranges.
TH_LEVEL_TO_SCORE = {
    "VLO": 12.5,
    "LOW": 37.5,
    "MED": 62.5,
    "HIG": 87.5,
}

# ThinkHazard mnemonic → NRI hazard code
TH_TO_NRI = {
    "EQ": "ERQK",
    "FL": "IFLD",
    "UF": "IFLD",   # merges with FL into IFLD
    "CF": "CFLD",
    "TS": "TSUN",
    "LS": "LNDS",
    "CY": "HRCN",
    "EH": "HWAV",
    "DG": "DRGT",
    "VO": "VLCN",
    "WF": "WFIR",
}

print("Mapping defined. Example conversion for CA division report:")
for entry in ca_hazards:
    th_code = entry["hazardtype"]["mnemonic"]
    nri_code = TH_TO_NRI.get(th_code)
    score = TH_LEVEL_TO_SCORE.get(entry["hazardlevel"]["mnemonic"])
    if nri_code:
        print(f"  {th_code} → {nri_code}_risk_score = {score}")

## 5. What Google Places v2 returns for CA/MX addresses

The plugin calls Google Places v2 autocomplete and reduces the result to:

```js
locData.county  = addr.administrative_area_level_2   // text name
locData.state   = addr.administrative_area_level_1   // text name
locData.st      = addr.administrative_area_level_1_short  // postal code (e.g. "ON", "JAL")
locData.country = addr.country                       // text ("Canada", "Mexico")
locData.fips    = null  // set later by FCC API — remains null for non-US
```

Below is a representative Places v2 response shape for a Canadian and a Mexican
address. These are static examples (Places v2 requires a browser API key and cannot
be called from a server-side notebook), annotated to show what is present and absent.

In [ ]:
# Representative Places v2 addressComponents for "123 Queen St W, Toronto, ON, Canada"
places_ca_example = {
    "displayName":        "123 Queen St W, Toronto, ON, Canada",
    "formattedAddress":   "123 Queen St W, Toronto, ON M5H 2M9, Canada",
    "location":           {"lat": 43.6487, "lng": -79.3835},  # ← present
    "addressComponents": [
        {"types": ["street_number"],                       "longText": "123",     "shortText": "123"},
        {"types": ["route"],                               "longText": "Queen Street West", "shortText": "Queen St W"},
        {"types": ["locality"],                            "longText": "Toronto",  "shortText": "Toronto"},
        # administrative_area_level_2 is the census division name — present as TEXT only
        {"types": ["administrative_area_level_2"],         "longText": "Toronto Division", "shortText": "Toronto Div."},
        # administrative_area_level_1_short is the postal abbreviation
        {"types": ["administrative_area_level_1"],         "longText": "Ontario",  "shortText": "ON"},
        {"types": ["country"],                             "longText": "Canada",   "shortText": "CA"},
        {"types": ["postal_code"],                         "longText": "M5H 2M9",  "shortText": "M5H 2M9"},
    ],
    # ← NOT present: any numeric GADM code, ThinkHazard code, or census division ID
}

# Representative Places v2 response for "Av. Reforma 222, Cuauhtémoc, CDMX, Mexico"
places_mx_example = {
    "displayName":        "Av. Paseo de la Reforma 222, Cuauhtémoc, CDMX",
    "formattedAddress":   "Av. Paseo de la Reforma 222, Cuauhtémoc, 06500 Ciudad de México, CDMX, Mexico",
    "location":           {"lat": 19.4284, "lng": -99.1644},  # ← present
    "addressComponents": [
        {"types": ["route"],                               "longText": "Avenida Paseo de la Reforma", "shortText": "Av. Paseo de la Reforma"},
        # administrative_area_level_2 = alcaldía/municipio name — TEXT only, accent-sensitive
        {"types": ["administrative_area_level_2"],         "longText": "Cuauhtémoc",  "shortText": "Cuauhtémoc"},
        {"types": ["administrative_area_level_1"],         "longText": "Ciudad de México", "shortText": "CDMX"},
        {"types": ["country"],                             "longText": "Mexico",      "shortText": "MX"},
        {"types": ["postal_code"],                         "longText": "06500",        "shortText": "06500"},
    ],
    # ← NOT present: INEGI municipio code, GADM code, ThinkHazard code
}

print("CA: administrative_area_level_2 =",
      next(c["longText"] for c in places_ca_example["addressComponents"]
           if "administrative_area_level_2" in c["types"]))
print("MX: administrative_area_level_2 =",
      next(c["longText"] for c in places_mx_example["addressComponents"]
           if "administrative_area_level_2" in c["types"]))
print()
print("What's missing for ThinkHazard lookup:")
print("  - GADM numeric division ID  (not in any Places v2 field)")
print("  - ThinkHazard numeric code  (not in any Places v2 field)")
print("  - INEGI municipio code (MX) (not in any Places v2 field)")
print("  - StatCan census div code (CA) (not in any Places v2 field)")

## 6. FCC API — live demonstration that it is US-only

The FCC Area API is the bridge used for US users: it converts a lat/lng into a
county FIPS code. The cells below call it with coordinates in **Ontario, Canada**
and **Mexico City** to show that it returns no useful data outside the US.

In [ ]:
# Toronto, Ontario — clearly outside the US
ca_lat, ca_lng = 43.6487, -79.3835
fcc_ca = requests.get(
    FCC_BASE,
    params={"latitude": ca_lat, "longitude": ca_lng, "format": "json"},
    timeout=10,
)
print(f"FCC API — Toronto, ON  (status {fcc_ca.status_code}):")
pprint.pprint(fcc_ca.json())

In [ ]:
# Mexico City
mx_lat, mx_lng = 19.4284, -99.1644
fcc_mx = requests.get(
    FCC_BASE,
    params={"latitude": mx_lat, "longitude": mx_lng, "format": "json"},
    timeout=10,
)
print(f"FCC API — Mexico City  (status {fcc_mx.status_code}):")
pprint.pprint(fcc_mx.json())

## 7. The lookup gap

The chain of problems:

```
Google Places v2
  ↓  gives us:
  • lat/lng
  • administrative_area_level_1_short  ("ON", "CDMX")  → enough to pick {ST}.csv
  • administrative_area_level_2 NAME   ("Toronto Division", "Cuauhtémoc")  → TEXT ONLY

ThinkHazard
  ↓  needs:
  • numeric division code              (e.g. 38471)  — no API to resolve by lat/lng or name

FCC Area API
  ↓  only works for:
  • US lat/lng → county FIPS code
  • Returns no useful data for CA or MX coordinates (demonstrated above)
```

### Why name-matching alone is unreliable

Even if we search the `admindiv_hazardsets` dump by `name` string, the match rate
is poor due to:

- Accent discrepancies (`Cuauhtémoc` vs `Cuauhtemoc`)
- Abbreviations (`Toronto Division` vs `Toronto`)
- Places v2 returning the locality name (`Toronto`) rather than the census division name
- Multiple countries sharing municipio/division names

In [ ]:
# Demonstrate name-match failures using the division data fetched earlier.
# These are the strings Places v2 would actually give us:
places_level2_examples = [
    ("Canada",  "Ontario",           "Toronto"),         # Places v2 often returns locality, not division
    ("Canada",  "Ontario",           "Toronto Division"), # Sometimes has the division suffix
    ("Canada",  "British Columbia",  "Greater Vancouver"),
    ("Canada",  "British Columbia",  "Metro Vancouver"),  # Colloquial name, not ThinkHazard name
    ("Mexico",  "Ciudad de México",  "Cuauhtémoc"),       # Accented
    ("Mexico",  "Ciudad de México",  "Cuauhtemoc"),       # Non-accented variant
    ("Mexico",  "Jalisco",           "Guadalajara"),
]

# Build lookup from (country, level_2, name) using the EQ division data
div_lookup = {
    (d["level_1"], d["level_2"], d["name"]): d["code"]
    for d in all_divisions
}

print(f"{'Places v2 input':<50}  {'ThinkHazard code':>20}")
print("-" * 72)
for country, province, name in places_level2_examples:
    key = (country, province, name)
    code = div_lookup.get(key, "NO MATCH")
    print(f"  ({country}, {province}, {name!r:<30})  → {code}")

## 8. What would close the gap

There are three viable paths, in order of implementation effort:

### Option A — Use ThinkHazard at CSV-generation time (recommended)

Generate the CA/MX CSV files in the notebook using ThinkHazard data, then add a
`division_name` column (normalized/slug) alongside `county_fips`. The JS plugin
then row-matches on the normalized `administrative_area_level_2` string instead of
FIPS. This requires:

1. Fetching `admindiv_hazardsets/{haz}.json` for all 11 hazards → one `code`→`score` map
2. Building a per-province/state CSV with rows keyed by division name slug
3. Minor JS change: for `country !== "US"`, match by `slug(administrative_area_level_2)` instead of FIPS

**Name-match coverage:** imperfect but improvable with a normalization table for known
Places v2 quirks (e.g. `Greater Vancouver` → `Greater Vancouver Regional District`).

### Option B — Add a lat/lng → GADM reverse-geocode step client-side

Services that return GADM admin-2 codes from lat/lng:

- **[GeoBoundaries](https://www.geoboundaries.org/)** — public, free, restful: `GET /api/point?longitude=&latitude=&adm=2`
- **[GADM reverse geocode via overpass/nominatim]** — complex, multi-step

This adds a JS round trip before CSV lookup but gives exact code resolution.

### Option C — Pre-build a lat/lng bounding-box index

Download GADM admin-2 GeoJSON for CA and MX, build a point-in-polygon lookup
table indexed by bounding box, and serve it as a small lookup JSON alongside
the risk CSVs. Eliminates the runtime API call but adds ~15 MB of static data.

---

**Next step:** implement Option A in `CA-MX_disaster_risk_analysis.ipynb`
and the corresponding JS change in `emfn-behavior-plugin.js`.